In [10]:
# =========================================================
# 模式 vs 觀測：2004-2024 三個溫度變數前處理完整版本
# =========================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd


# =========================================================
# 0. 基本設定
# =========================================================
POI_LAT = 25.0190099143756
POI_LON = 121.53137177116402

START_YEAR = 2004
END_YEAR = 2024

TARGET_SSPS = ["ssp245", "ssp370", "ssp585"]

MIN_COVERAGE = 0.90
REQUIRE_CONTINUOUS = True

obs_dirs = {
    "tmean": Path(r"C:/觀測_日資料_臺北市_平均溫"),
    "tmax":  Path(r"C:/觀測_日資料_臺北市_最高溫"),
    "tmin":  Path(r"C:/觀測_日資料_臺北市_最低溫"),
}

model_dirs = {
    "tmean": Path(r"C:/AR5_統計降尺度_日資料_臺北市_平均溫"),
    "tmax":  Path(r"C:/AR5_統計降尺度_日資料_臺北市_最高溫"),
    "tmin":  Path(r"C:/AR5_統計降尺度_日資料_臺北市_最低溫"),
}


# =========================================================
# 1. 基本工具
# =========================================================
def read_csv_flexible(csv_path: Path) -> pd.DataFrame:
    """
    安全讀取 CSV：
    - 處理 utf-8-sig / BOM
    - 若被讀成單一欄，自動再猜分隔符
    - 清理欄名的空白、單引號、雙引號
    """
    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    # 若整份被讀成單一欄，改用自動猜分隔符
    if len(df.columns) == 1:
        df = pd.read_csv(csv_path, encoding="utf-8-sig", sep=None, engine="python")

    # 清理欄名
    clean_cols = []
    for c in df.columns:
        c = str(c).strip()
        c = c.replace("\ufeff", "")
        c = c.replace("'", "")
        c = c.replace('"', "")
        clean_cols.append(c)

    df.columns = clean_cols
    return df


def remove_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.columns.astype(str).str.contains(r"^Unnamed")].copy()


def get_date_columns(columns, start_year=2004, end_year=2024):
    out = []
    for c in columns:
        c = str(c)
        if re.fullmatch(r"\d{8}", c):
            y = int(c[:4])
            if start_year <= y <= end_year:
                out.append(c)
    return sorted(out)


def to_datetime_index(idx):
    return pd.to_datetime(pd.Index(idx).astype(str), format="%Y%m%d", errors="coerce")


def ensure_lat_lon(df: pd.DataFrame):
    if "LON" not in df.columns or "LAT" not in df.columns:
        raise KeyError(f"缺少 LON/LAT 欄位，實際欄位為：{df.columns.tolist()[:20]}")
    return df


def find_nearest_row(df: pd.DataFrame, target_lat: float, target_lon: float):
    df = ensure_lat_lon(df)
    dist2 = (df["LAT"] - target_lat) ** 2 + (df["LON"] - target_lon) ** 2
    idx = dist2.idxmin()
    return df.loc[idx].copy()


def estimate_grid_spacing(df: pd.DataFrame, target_lat: float, target_lon: float):
    df = ensure_lat_lon(df)

    nearest = find_nearest_row(df, target_lat, target_lon)
    lat0 = float(nearest["LAT"])
    lon0 = float(nearest["LON"])

    unique_lats = np.sort(df["LAT"].dropna().unique())
    unique_lons = np.sort(df["LON"].dropna().unique())

    lat_neighbors = unique_lats[np.abs(unique_lats - lat0) > 0]
    lon_neighbors = unique_lons[np.abs(unique_lons - lon0) > 0]

    dlat = float(np.min(np.abs(lat_neighbors - lat0))) if len(lat_neighbors) else 0.05
    dlon = float(np.min(np.abs(lon_neighbors - lon0))) if len(lon_neighbors) else 0.05

    return lat0, lon0, dlat, dlon


def calc_coverage_ratio(dt_index, start_year=2004, end_year=2024):
    full_range = pd.date_range(f"{start_year}-01-01", f"{end_year}-12-31", freq="D")
    return len(dt_index) / len(full_range) if len(full_range) else 0.0


def is_series_continuous(dt_index):
    if len(dt_index) == 0:
        return False
    idx = pd.DatetimeIndex(dt_index).sort_values()
    expected = pd.date_range(idx.min(), idx.max(), freq="D")
    return len(idx) == len(expected) and idx.equals(expected)


def validate_common_period(common_index, start_year=2004, end_year=2024,
                           min_coverage=0.90, require_continuous=True):
    coverage = calc_coverage_ratio(common_index, start_year, end_year)
    continuous = is_series_continuous(common_index)

    ok = coverage >= min_coverage
    if require_continuous:
        ok = ok and continuous

    return {
        "ok": ok,
        "coverage": coverage,
        "continuous": continuous,
        "n_common_days": len(common_index)
    }


# =========================================================
# 2. 檔名解析
# 預期格式：
# ..._historical_MODEL_2004.csv
# ..._ssp245_MODEL_2015.csv
# =========================================================
def parse_model_filename(file_path: Path):
    stem = file_path.stem
    m = re.search(r"(historical|ssp\d{3})_(.+?)_(\d{4})$", stem, flags=re.IGNORECASE)
    if not m:
        return None

    return {
        "scenario": m.group(1).lower(),
        "model": m.group(2),
        "year": int(m.group(3)),
        "path": file_path
    }


def parse_obs_filename(file_path: Path):
    stem = file_path.stem
    m = re.search(r"(\d{4})$", stem)
    if not m:
        return None
    return int(m.group(1))


# =========================================================
# 3. 建立檔案索引
# =========================================================
def build_model_file_index(var_dir: Path, start_year=2004, end_year=2024):
    rows = []

    for f in var_dir.rglob("*"):
        if not f.is_file():
            continue
        if f.suffix.lower() != ".csv":
            continue

        parsed = parse_model_filename(f)
        if parsed is None:
            continue

        if start_year <= parsed["year"] <= end_year:
            rows.append(parsed)

    if not rows:
        return pd.DataFrame(columns=["scenario", "model", "year", "path"])

    df = pd.DataFrame(rows).sort_values(["model", "scenario", "year"]).reset_index(drop=True)
    return df


def build_obs_file_index(obs_dir: Path, start_year=2004, end_year=2024):
    rows = []

    for f in obs_dir.rglob("*"):
        if not f.is_file():
            continue
        if f.suffix.lower() != ".csv":
            continue

        year = parse_obs_filename(f)
        if year is None:
            continue

        if start_year <= year <= end_year:
            rows.append({
                "year": year,
                "path": f
            })

    if not rows:
        return pd.DataFrame(columns=["year", "path"])

    df = pd.DataFrame(rows).sort_values("year").drop_duplicates("year").reset_index(drop=True)
    return df


# =========================================================
# 4. 讀單一模式檔：最近格點
# =========================================================
def read_model_one_file_nearest(csv_path: Path, target_lat, target_lon,
                                start_year=2004, end_year=2024):
    df = read_csv_flexible(csv_path)
    df = remove_unnamed_columns(df)
    df = ensure_lat_lon(df)

    nearest = find_nearest_row(df, target_lat, target_lon)
    date_cols = get_date_columns(df.columns, start_year, end_year)

    if not date_cols:
        return pd.Series(dtype=float), None, None

    s = nearest[date_cols].copy()
    s.index = to_datetime_index(s.index)
    s = pd.to_numeric(s, errors="coerce")
    s = s[~s.index.isna()].sort_index()

    model_lat = float(nearest["LAT"])
    model_lon = float(nearest["LON"])

    return s, model_lat, model_lon


# =========================================================
# 5. 讀單一年觀測檔：模式格內觀測格點平均
# =========================================================
def read_obs_one_year_spatial_mean(csv_path: Path, model_lat, model_lon, dlat, dlon,
                                   start_year=2004, end_year=2024):
    df = read_csv_flexible(csv_path)
    df = remove_unnamed_columns(df)
    df = ensure_lat_lon(df)

    lat_min = model_lat - dlat / 2
    lat_max = model_lat + dlat / 2
    lon_min = model_lon - dlon / 2
    lon_max = model_lon + dlon / 2

    sub = df[
        (df["LAT"] >= lat_min) & (df["LAT"] <= lat_max) &
        (df["LON"] >= lon_min) & (df["LON"] <= lon_max)
    ].copy()

    # 若框不到，就退回最近觀測格點
    if sub.empty:
        nearest = find_nearest_row(df, model_lat, model_lon)
        sub = pd.DataFrame([nearest])

    date_cols = get_date_columns(sub.columns, start_year, end_year)
    if not date_cols:
        return pd.Series(dtype=float)

    s = sub[date_cols].mean(axis=0)
    s.index = to_datetime_index(s.index)
    s = pd.to_numeric(s, errors="coerce")
    s = s[~s.index.isna()].sort_index()

    return s


# =========================================================
# 6. 組裝模式與觀測時間序列
# 2004-2014 -> historical
# 2015-2024 -> 指定 ssp
# =========================================================
def load_model_series_from_index(index_df, model_name, ssp, target_lat, target_lon,
                                 start_year=2004, end_year=2024):
    if index_df.empty:
        return pd.Series(dtype=float), None, None, []

    selected_rows = []

    for year in range(start_year, end_year + 1):
        if year <= 2014:
            sub = index_df[
                (index_df["model"] == model_name) &
                (index_df["scenario"] == "historical") &
                (index_df["year"] == year)
            ]
        else:
            sub = index_df[
                (index_df["model"] == model_name) &
                (index_df["scenario"] == ssp) &
                (index_df["year"] == year)
            ]

        if not sub.empty:
            selected_rows.append(sub.iloc[0])

    if not selected_rows:
        return pd.Series(dtype=float), None, None, []

    series_list = []
    source_files = []
    chosen_model_lat = None
    chosen_model_lon = None

    for row in selected_rows:
        s, model_lat, model_lon = read_model_one_file_nearest(
            row["path"], target_lat, target_lon, start_year, end_year
        )

        if s.empty:
            continue

        s = s[s.index.year == row["year"]]
        if s.empty:
            continue

        series_list.append(s)
        source_files.append(row["path"])

        if chosen_model_lat is None:
            chosen_model_lat = model_lat
            chosen_model_lon = model_lon

    if not series_list:
        return pd.Series(dtype=float), None, None, []

    out = pd.concat(series_list).sort_index()
    out = out[~out.index.duplicated(keep="first")]
    return out, chosen_model_lat, chosen_model_lon, source_files


def load_obs_series_from_index(obs_index_df, model_lat, model_lon, dlat, dlon,
                               start_year=2004, end_year=2024):
    if obs_index_df.empty:
        return pd.Series(dtype=float)

    series_list = []

    for year in range(start_year, end_year + 1):
        sub = obs_index_df[obs_index_df["year"] == year]
        if sub.empty:
            continue

        f = sub.iloc[0]["path"]
        s = read_obs_one_year_spatial_mean(
            f, model_lat, model_lon, dlat, dlon, start_year, end_year
        )

        if s.empty:
            continue

        s = s[s.index.year == year]
        if not s.empty:
            series_list.append(s)

    if not series_list:
        return pd.Series(dtype=float)

    out = pd.concat(series_list).sort_index()
    out = out[~out.index.duplicated(keep="first")]
    return out


# =========================================================
# 7. 主流程
# =========================================================
def prepare_temperature_data(
    obs_dirs,
    model_dirs,
    variables=("tmean", "tmax", "tmin"),
    target_ssps=("ssp245", "ssp370", "ssp585"),
    target_lat=25.0190099143756,
    target_lon=121.53137177116402,
    start_year=2004,
    end_year=2024,
    min_coverage=0.90,
    require_continuous=True
):
    temp_data = {}
    availability_rows = []

    for var in variables:
        print(f"\n==============================")
        print(f"[PROCESS] variable = {var}")
        print(f"==============================")

        var_model_dir = model_dirs[var]
        var_obs_dir = obs_dirs[var]

        print(f"[MODEL DIR] {var_model_dir}")
        print(f"[OBS DIR]   {var_obs_dir}")

        if not var_model_dir.exists():
            print(f"[WARN] model dir not found: {var_model_dir}")
            temp_data[var] = {}
            continue

        if not var_obs_dir.exists():
            print(f"[WARN] obs dir not found: {var_obs_dir}")
            temp_data[var] = {}
            continue

        model_index = build_model_file_index(var_model_dir, start_year, end_year)
        obs_index = build_obs_file_index(var_obs_dir, start_year, end_year)

        print(f"[INFO] model files found: {len(model_index)}")
        print(f"[INFO] obs files found  : {len(obs_index)}")

        if model_index.empty:
            print(f"[WARN] {var} 沒有掃到模式檔")
            temp_data[var] = {}
            continue

        if obs_index.empty:
            print(f"[WARN] {var} 沒有掃到觀測檔")
            temp_data[var] = {}
            continue

        models = sorted(model_index["model"].unique())
        print(f"[INFO] models found: {len(models)}")
        print(models)

        sample_file = model_index.iloc[0]["path"]
        sample_df = read_csv_flexible(sample_file)
        sample_df = remove_unnamed_columns(sample_df)
        sample_df = ensure_lat_lon(sample_df)

        _, _, dlat, dlon = estimate_grid_spacing(sample_df, target_lat, target_lon)
        print(f"[INFO] estimated grid spacing: dlat={dlat:.4f}, dlon={dlon:.4f}")

        temp_data[var] = {}

        for model_name in models:
            temp_data[var][model_name] = {}

            for ssp in target_ssps:
                print(f"\n--- {var} | {model_name} | {ssp} ---")

                model_ts, model_lat, model_lon, source_files = load_model_series_from_index(
                    model_index, model_name, ssp, target_lat, target_lon, start_year, end_year
                )

                if model_ts.empty or model_lat is None:
                    print("[DROP] no model series")
                    availability_rows.append({
                        "variable": var,
                        "model": model_name,
                        "ssp": ssp,
                        "status": "drop_no_model_data",
                        "coverage": np.nan,
                        "continuous": np.nan,
                        "n_common_days": 0,
                        "model_lat": np.nan,
                        "model_lon": np.nan,
                    })
                    continue

                obs_ts = load_obs_series_from_index(
                    obs_index, model_lat, model_lon, dlat, dlon, start_year, end_year
                )

                if obs_ts.empty:
                    print("[DROP] no obs series")
                    availability_rows.append({
                        "variable": var,
                        "model": model_name,
                        "ssp": ssp,
                        "status": "drop_no_obs_data",
                        "coverage": np.nan,
                        "continuous": np.nan,
                        "n_common_days": 0,
                        "model_lat": model_lat,
                        "model_lon": model_lon,
                    })
                    continue

                common_index = model_ts.index.intersection(obs_ts.index).sort_values()

                if len(common_index) == 0:
                    print("[DROP] no common dates")
                    availability_rows.append({
                        "variable": var,
                        "model": model_name,
                        "ssp": ssp,
                        "status": "drop_no_common_dates",
                        "coverage": 0.0,
                        "continuous": False,
                        "n_common_days": 0,
                        "model_lat": model_lat,
                        "model_lon": model_lon,
                    })
                    continue

                check = validate_common_period(
                    common_index,
                    start_year=start_year,
                    end_year=end_year,
                    min_coverage=min_coverage,
                    require_continuous=require_continuous
                )

                if not check["ok"]:
                    print(f"[DROP] coverage={check['coverage']:.3f}, continuous={check['continuous']}, n={check['n_common_days']}")
                    availability_rows.append({
                        "variable": var,
                        "model": model_name,
                        "ssp": ssp,
                        "status": "drop_incomplete",
                        "coverage": check["coverage"],
                        "continuous": check["continuous"],
                        "n_common_days": check["n_common_days"],
                        "model_lat": model_lat,
                        "model_lon": model_lon,
                    })
                    continue

                model_common = model_ts.loc[common_index]
                obs_common = obs_ts.loc[common_index]

                out_df = pd.DataFrame({
                    "obs": obs_common.values,
                    "model": model_common.values,
                }, index=common_index)

                out_df["bias"] = out_df["model"] - out_df["obs"]

                temp_data[var][model_name][ssp] = {
                    "model_lat": model_lat,
                    "model_lon": model_lon,
                    "coverage": check["coverage"],
                    "continuous": check["continuous"],
                    "n_common_days": check["n_common_days"],
                    "source_files": source_files,
                    "data": out_df
                }

                print(f"[KEEP] coverage={check['coverage']:.3f}, continuous={check['continuous']}, n={check['n_common_days']}")

                availability_rows.append({
                    "variable": var,
                    "model": model_name,
                    "ssp": ssp,
                    "status": "keep",
                    "coverage": check["coverage"],
                    "continuous": check["continuous"],
                    "n_common_days": check["n_common_days"],
                    "model_lat": model_lat,
                    "model_lon": model_lon,
                })

    availability_df = pd.DataFrame(availability_rows)
    return temp_data, availability_df


# =========================================================
# 8. 執行
# =========================================================
temp_data, availability_df = prepare_temperature_data(
    obs_dirs=obs_dirs,
    model_dirs=model_dirs,
    variables=("tmean", "tmax", "tmin"),
    target_ssps=TARGET_SSPS,
    target_lat=POI_LAT,
    target_lon=POI_LON,
    start_year=START_YEAR,
    end_year=END_YEAR,
    min_coverage=MIN_COVERAGE,
    require_continuous=REQUIRE_CONTINUOUS
)

print("\n===== availability_df =====")
if availability_df.empty:
    print("沒有任何結果，請檢查路徑或檔名格式")
else:
    print(availability_df.sort_values(["variable", "model", "ssp"]).to_string(index=False))


# =========================================================
# 9. summary
# =========================================================
def calc_metrics(df):
    diff = df["model"] - df["obs"]
    return pd.Series({
        "bias_mean": diff.mean(),
        "mae": diff.abs().mean(),
        "rmse": np.sqrt((diff ** 2).mean()),
        "n_days": len(df)
    })

summary_rows = []

for var, model_dict in temp_data.items():
    for model_name, ssp_dict in model_dict.items():
        for ssp, info in ssp_dict.items():
            df = info["data"]
            met = calc_metrics(df)
            met["variable"] = var
            met["model"] = model_name
            met["ssp"] = ssp
            met["coverage"] = info["coverage"]
            met["continuous"] = info["continuous"]
            summary_rows.append(met)

summary_df = pd.DataFrame(summary_rows)

print("\n===== summary_df =====")
if summary_df.empty:
    print("目前沒有通過篩選的模式")
else:
    summary_df = summary_df[
        ["variable", "model", "ssp", "bias_mean", "mae", "rmse", "n_days", "coverage", "continuous"]
    ].sort_values(["variable", "model", "ssp"])
    print(summary_df.to_string(index=False))


# =========================================================
# 10. 範例：如何取出某組資料
# =========================================================
# df_example = temp_data["tmean"]["TaiESM1"]["ssp245"]["data"]
# print(df_example.head())


[PROCESS] variable = tmean
[MODEL DIR] C:\AR5_統計降尺度_日資料_臺北市_平均溫
[OBS DIR]   C:\觀測_日資料_臺北市_平均溫
[INFO] model files found: 68
[INFO] obs files found  : 21
[INFO] models found: 34
['ACCESS1-0', 'ACCESS1-3', 'BNU-ESM', 'CCSM4', 'CESM1-BGC', 'CESM1-CAM5', 'CMCC-CESM', 'CMCC-CM', 'CMCC-CMS', 'CNRM-CM5', 'CSIRO-Mk3-6-0', 'CanESM2', 'EC-EARTH', 'FGOALS-g2', 'GFDL-CM3', 'GFDL-ESM2G', 'GFDL-ESM2M', 'HadGEM2-AO', 'HadGEM2-CC', 'HadGEM2-ES', 'IPSL-CM5A-LR', 'IPSL-CM5A-MR', 'IPSL-CM5B-LR', 'MIROC-ESM', 'MIROC-ESM-CHEM', 'MIROC5', 'MPI-ESM-LR', 'MPI-ESM-MR', 'MRI-CGCM3', 'MRI-ESM1', 'NorESM1-M', 'bcc-csm1-1', 'bcc-csm1-1-m', 'inmcm4']
[INFO] estimated grid spacing: dlat=0.0500, dlon=0.0500

--- tmean | ACCESS1-0 | ssp245 ---
[DROP] coverage=0.095, continuous=False, n=730

--- tmean | ACCESS1-0 | ssp370 ---
[DROP] coverage=0.095, continuous=False, n=730

--- tmean | ACCESS1-0 | ssp585 ---
[DROP] coverage=0.095, continuous=False, n=730

--- tmean | ACCESS1-3 | ssp245 ---
[DROP] coverage=0.095, continu

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =====================================================
# 路徑設定
# =====================================================

obs_dirs = {
    "tmean": Path(r"C:/觀測_日資料_臺北市_平均溫"),
    "tmax":  Path(r"C:/觀測_日資料_臺北市_最高溫"),
    "tmin":  Path(r"C:/觀測_日資料_臺北市_最低溫"),
}

model_dirs = {
    "tmean": Path(r"C:/AR5_統計降尺度_日資料_臺北市_平均溫"),
    "tmax":  Path(r"C:/AR5_統計降尺度_日資料_臺北市_最高溫"),
    "tmin":  Path(r"C:/AR5_統計降尺度_日資料_臺北市_最低溫"),
}

TARGET_SSPS = ["ssp245", "ssp370", "ssp585"]

START_YEAR = 2004
END_YEAR   = 2024

POI_LAT = 25.0190
POI_LON = 121.5313


# =====================================================
# 基本工具
# =====================================================

def read_csv_flexible(path):
    df = pd.read_csv(path, encoding="utf-8-sig")

    if len(df.columns) == 1:
        df = pd.read_csv(path, encoding="utf-8-sig", sep=None, engine="python")

    # 🔥 清理欄名（解決 KeyError）
    df.columns = [
        str(c).strip().replace("'", "").replace('"', "").replace("\ufeff", "")
        for c in df.columns
    ]

    return df


def remove_unnamed_columns(df):
    return df.loc[:, ~df.columns.str.contains("^Unnamed")]


def get_date_columns(columns, start_year, end_year):
    return [
        c for c in columns
        if re.match(r"\d{8}", str(c))
        and start_year <= int(c[:4]) <= end_year
    ]


def find_nearest_value(df, lat, lon):
    df["dist"] = np.sqrt((df["LAT"] - lat)**2 + (df["LON"] - lon)**2)
    return df.loc[df["dist"].idxmin()]


# =====================================================
# 建立 model index
# =====================================================

def build_model_index(model_dir):
    rows = []

    for f in model_dir.rglob("*.csv"):
        name = f.name.lower()

        model = f.parent.name.lower()

        if "ssp" in name:
            scenario = re.search(r"(ssp\d+)", name).group(1)
        elif "historical" in name:
            scenario = "historical"
        else:
            continue

        year_match = re.search(r"(19|20)\d{2}", name)
        if not year_match:
            continue

        year = int(year_match.group())

        rows.append({
            "model": model,
            "scenario": scenario,
            "year": year,
            "path": f
        })

    return pd.DataFrame(rows)


# =====================================================
# 🔥 檢測並排除 360-day 模式
# =====================================================

def detect_360_day_models(model_index):

    bad_models = []

    for model in model_index["model"].unique():

        sub = model_index[model_index["model"] == model].head(3)

        day_counts = []

        for _, row in sub.iterrows():
            df = read_csv_flexible(row["path"])
            df = remove_unnamed_columns(df)

            date_cols = get_date_columns(df.columns, START_YEAR, END_YEAR)
            day_counts.append(len(date_cols))

        if len(day_counts) > 0 and all(x == 360 for x in day_counts):
            bad_models.append(model)

    return bad_models


# =====================================================
# 主流程
# =====================================================

def prepare_data():

    all_data = {}

    for var in ["tmean", "tmax", "tmin"]:

        print(f"\n===== {var} =====")

        model_index = build_model_index(model_dirs[var])

        # 🔥 排除 360-day
        bad_models = detect_360_day_models(model_index)

        if bad_models:
            print(f"[INFO] 排除 360-day 模式: {bad_models}")
            model_index = model_index[~model_index["model"].isin(bad_models)]

        if model_index.empty:
            print("[WARN] 沒有可用模式")
            continue

        models = sorted(model_index["model"].unique())
        print(f"[INFO] models: {models}")

        var_data = {}

        for model in models:

            model_df = model_index[model_index["model"] == model]

            all_series = []

            for _, row in model_df.iterrows():

                df = read_csv_flexible(row["path"])
                df = remove_unnamed_columns(df)

                point = find_nearest_value(df, POI_LAT, POI_LON)

                date_cols = get_date_columns(df.columns, START_YEAR, END_YEAR)

                values = point[date_cols].astype(float).values
                all_series.append(values)

            if len(all_series) == 0:
                continue

            var_data[model] = np.concatenate(all_series)

        # =========================
        # 觀測資料
        # =========================

        obs_files = list(obs_dirs[var].rglob("*.csv"))

        obs_series = []

        for f in obs_files:
            df = read_csv_flexible(f)
            df = remove_unnamed_columns(df)

            point = find_nearest_value(df, POI_LAT, POI_LON)

            date_cols = get_date_columns(df.columns, START_YEAR, END_YEAR)

            values = point[date_cols].astype(float).values
            obs_series.append(values)

        obs_series = np.concatenate(obs_series)

        all_data[var] = {
            "model": var_data,
            "obs": obs_series
        }

    return all_data


# =====================================================
# 執行
# =====================================================

data = prepare_data()

print("\n✅ DONE")


===== tmean =====
[INFO] models: ['ar5_統計降尺度_日資料_臺北市_平均溫']

===== tmax =====
[INFO] models: ['ar5_統計降尺度_日資料_臺北市_最高溫']

===== tmin =====
[INFO] models: ['ar5_統計降尺度_日資料_臺北市_最低溫']

✅ DONE
